# Learning-rate comparison — saved-model pilot
Run this after notebook 18 finishes. Both arms copy the latest durable campaign model at Start, including Adam and RNG state. Your main campaign stays untouched.

**Same exact 176 decks, six threads, four collectors.** Current rate 0.0001 versus 0.00005; 128 games per arm in 64-game batches. Common frozen evaluation opponents and seeds: 32 cases per opponent, after each arm. Same self-play mixture, KL guard and lessons.

90-minute soft budget across the experiment; a batch or evaluation already underway can exceed it. If either arm stops early, the comparison is inconclusive and all partial checkpoints are retained. This small single-seed pilot is evidence for a follow-up, not proof of a winner. Re-running Start creates a new isolated pilot, not a continuation.

The checks below do not train or freeze today's unfinished checkpoint. Start refuses while another run is active, or if the latest model needs a reviewed engine migration/failure repair.

In [ ]:
from pathlib import Path
import subprocess, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'tools/train_neural.py').exists())
PYTHON = ROOT / '.venv/bin/python'
TOOL = ROOT / 'tools/learning_rate_comparison.py'
check = subprocess.run([str(PYTHON), str(TOOL)], cwd=ROOT, text=True, capture_output=True)
print(check.stdout or check.stderr)
check.check_returncode()


## Start — owner only
Leave this cell unrun until the campaign ends. It will refuse overlapping training. Outputs show batch saves, PPO diagnostics and evaluations. No model is automatically promoted.

In [ ]:
process = subprocess.Popen([str(PYTHON), str(TOOL), '--train'], cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line.rstrip(), flush=True)
    status = process.wait()
except BaseException:
    process.terminate()
    process.wait()
    raise
if status:
    raise RuntimeError('Comparison did not complete; inspect the message above. Existing campaign progress is untouched.')
